> **교육 목표** 정확도만으로는 모델을 평가할 수 없음을 확인하고, 혼동행렬과 재현율로 불균형 데이터를 평가합니다.
>
> **핵심 내용** 고장 비율 → X·y와 시간 분할 → "모두 정상" 기준 모델 → 로지스틱 회귀·의사결정나무 → 세 모델 비교

In [1]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D3_01 기준 모델과 혼동행렬

> **오늘 질문:** 정확도 96%면 좋은 모델일까?

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

In [ ]:
# 정제가 끝난 분석 테이블 (시간 순으로 정렬되어 있음)
df = pd.read_parquet("../../data/factory_table.parquet")
df.head()

## 1. 고장은 얼마나 드문가 ✍️

`failure` 컬럼에서 정상(0)과 고장(1)의 건수를 셉니다.

In [ ]:
# 힌트: df["컬럼"].value_counts()

## 2. 입력(X)과 정답(y) 정하기 ✍️

입력은 센서 5개(`air_temp`, `process_temp`, `rpm`, `torque`, `tool_wear`), 정답은 고장 여부(`failure`)입니다.
고장 유형 컬럼(`twf` ~ `rnf`)은 정답을 그대로 담고 있어서 넣지 않습니다. (3교시에서 자세히)

> 변수 이름: `features`, `X`, `y`

## 3. 시간 순으로 나누기 ✍️

앞 80% 기간으로 학습하고, **마지막 20% 기간**으로 평가합니다.

> 변수 이름: `cut`, `X_train`, `X_test`, `y_train`, `y_test`

In [ ]:
# 힌트: X.iloc[:cut] 은 앞에서 cut개, X.iloc[cut:] 은 나머지

## 4. 기준 모델: "모두 정상" ✍️

무조건 가장 많은 값(정상)으로 예측하는 모델입니다. 모든 모델은 이 기준선부터 넘어야 합니다.

> 변수 이름: 모델 `base`, 예측 `pred_base`

In [ ]:
# 힌트: DummyClassifier(strategy="most_frequent")

**Q1.** 정확도와 재현율은 각각 얼마인가요? 이 모델은 평가 기간의 고장을 몇 건 잡아냈나요?

> 답:


## 5. 로지스틱 회귀 ✍️

`classification_report`는 정밀도·재현율·F1을 한 번에 보여 줍니다. 아래 **고장** 줄을 보세요.

> 변수 이름: 모델 `lr`, 예측 `pred_lr`

In [ ]:
# 힌트: LogisticRegression(max_iter=1000), classification_report(정답, 예측)

혼동행렬로 네 칸을 확인합니다. 세로축(True label)은 실제, 가로축(Predicted label)은 예측입니다.
오른쪽 위는 오경보(FP), 왼쪽 아래는 놓친 고장(FN)입니다.

In [ ]:
# 힌트: ConfusionMatrixDisplay.from_predictions(정답, 예측)

## 6. 의사결정나무 ✍️

나무의 깊이를 5로 제한합니다. 사용법은 로지스틱 회귀와 같습니다. 혼동행렬까지 그려 보세요.

> 변수 이름: 모델 `tree`, 예측 `pred_tree`

In [ ]:
# 힌트: DecisionTreeClassifier(max_depth=5, random_state=42)

## 7. 세 모델 비교 📋

In [ ]:
preds = {"모두 정상": pred_base, "로지스틱 회귀": pred_lr, "의사결정나무": pred_tree}

rows = []
for name, p in preds.items():
    rows.append({
        "모델": name,
        "정확도": accuracy_score(y_test, p),
        "정밀도": precision_score(y_test, p, zero_division=0),
        "재현율": recall_score(y_test, p),
    })
pd.DataFrame(rows).round(3)

## 정리 💬

**Q2.** 정확도로만 보면 세 모델의 차이가 큰가요? 재현율로 보면 어떤가요?

> 답:


**Q3.** 고장 1건을 놓치면 라인이 4시간 멈추고, 헛경보 1건은 점검 10분이면 끝납니다. 어떤 지표를 가장 중요하게 봐야 할까요?

> 답:
